# 03 — Новый эксперимент: 100 000 train + 100 000 test

В каждой точке всей сетки заново обучаем стратегию, затем проверяем на независимых
тестовых траекториях. Стоимость, SE и 95%-й доверительный интервал сохраняются сразу.
Старый запуск не используется. Выполните ячейки сверху вниз.


In [1]:
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
from osfbm.config import Config
from osfbm.experiment import ExperimentSettings, prepare_experiment, run_experiment, load_experiment
from osfbm.results import run_grid
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODE = "grid"  # совместимость со старым форматом; настройки ниже выбирают новый эксперимент


## Параметры нового эксперимента

100 000 траекторий полностью используются для обучения регрессий; 100 000 других —
для тестирования. Ещё 10 000 независимых валидационных траекторий выбирают действие
в нулевой момент **до теста**. Тестовое среднее не обрезается через max(0, V).

Внутри каждого H выборки общие для разных μ. Train, validation и test независимы.
`RUN_ID` фиксирован: повторный запуск продолжает вычисления. При изменении параметров
укажите новый RUN_ID. Полная сетка содержит 59 846 узлов и требует длительного расчёта.


In [ ]:
MODE = "fresh"
RUN_ID = "fresh_train20k_test20k_v1"
cfg = Config(
    T=1.0, n_fine=200, n_exercise=20,
    M_train=20_000, M_test=20_000,
    K=4, seed=20260918,
    mu_grid=tuple(np.round(np.arange(-4.0, 9.01, 0.01), 6)),
    H_grid=tuple(np.round(np.arange(0.06, 0.1, 0.02), 6)),
)
settings = ExperimentSettings(
    epsilon=0.03, confidence=0.95,
    batch_size=2_000, M_validation=10_000,
)
RUN_DIR = ROOT / "result_optimal_stopping" / "runs" / RUN_ID
print("Каталог нового эксперимента:", RUN_DIR)
print(f"Узлов: {len(cfg.H_grid)*len(cfg.mu_grid)}; train={cfg.M_train}; test={cfg.M_test}")


Каталог нового эксперимента: /Users/armkilikia/optimal-stopping-fbm/result_optimal_stopping/runs/fresh_train20k_test20k_v1
Узлов: 13010; train=20000; test=20000


## Запуск и продолжение

Для каждого узла сохраняются обученная стратегия и прогресс теста по блокам.
Готовые узлы не пересчитываются. Во время теста показывается число обработанных
траекторий; после завершения узла — среднее и 95%-й интервал стоимости.
Для продолжения используйте те же RUN_ID и параметры.


In [3]:
if MODE == "fresh":
    manifest = prepare_experiment(RUN_DIR, cfg, settings)
    _, saved, _ = load_experiment(RUN_DIR)
    with tqdm(total=manifest["family_size"], initial=int(saved.complete.sum()), desc="Узлы") as bar:
        def report(event):
            if event["stage"] == "node":
                bar.update(1)
                bar.set_postfix(H=event["H"], mu=event["mu"], V=f"{event['V']:.5f}",
                                CI=f"[{event['ci_low']:.5f}, {event['ci_high']:.5f}]")
            else:
                bar.set_postfix(stage=event["stage"], H=event["H"], mu=event["mu"], n=event.get("n", 0))
        manifest, nodes, boundaries = run_experiment(cfg, RUN_DIR, settings, progress=report)
    print("Готовых узлов:", int(nodes.complete.sum()))
    print("Откройте 04-grid-analysis.ipynb с тем же RUN_ID.")
elif MODE == "grid":
    nodes = run_grid(cfg, RUN_DIR)
else:
    raise ValueError("MODE должен быть fresh или grid")


Узлы:   0%|          | 0/13010 [00:00<?, ?it/s]

/Users/armkilikia/optimal-stopping-fbm/src/osfbm/vendor/Signature_computer.py:36: SyntaxWarning: invalid escape sequence '\i'
  """


KeyboardInterrupt: 

## Что означают интервалы

Интервал V оценивает погрешность независимого теста для новой обученной стратегии.
Для интервалов μ используется более широкая общая полоса по всей сетке.
Это не контроль ошибки относительно истинного оптимума: верхняя dual-оценка не добавлена.
Подробнее: [новый эксперимент и интервалы](../docs/06-fresh-experiment.md).
